In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_anthropic import ChatAnthropic
from pydantic import BaseModel, Field
import os
from langchain_core.tools import tool
from langgraph.graph import MessagesState
from langchain_core.messages import SystemMessage, HumanMessage, ToolMessage
from langchain_core.messages import ToolMessage
from typing import Literal
from IPython.display import Image, display
import pandas as pd
import ast


# Set API key manually
os.environ["ANTHROPIC_API_KEY"] = ""

In [19]:
llm = ChatAnthropic(
    model="claude-3-5-sonnet-latest",
    api_key=os.getenv("ANTHROPIC_API_KEY"),  # Use the environment variable
    temperature=0.2,
    max_tokens=1024
)

In [20]:
from typing import Union

class Term(BaseModel):
    ASPECT_TERM: Union[list, str] = Field(None, description="list of aspect terms of the given sentence")

structured_llm = llm.with_structured_output(Term)
  

In [21]:
ate_prompt = """You are Linguistic expert. I want you to find the ASPECT TERM of the review sentence.:
-Examples: 
1. After all that, they complained to me about the small tip.
[noaspectterm] 
2.Chow fun was dry; pork shu mai was more than usually greasy and had to share a table with loud and rude family. 
['Chow fun', 'pork shu mai']
3. We took advanatage of the half price sushi deal on saturday so it was well worth it.
['half price sushi deal']
Now find the ASPECT TERM of the following sentence:
"""

In [22]:
ate_16_test = pd.read_csv("/home/s6moakba/InstructABSA/Dataset/SemEval16/Test/Restaurants_Test.csv")

In [26]:
claude_pred = []
for i, row in ate_16_test.iterrows():
    text = row['raw_text']
    print(i)
    print(text)
    pred = structured_llm.invoke(ate_prompt + text)

    if isinstance(pred.ASPECT_TERM, str):
        try:
            pred.ASPECT_TERM = ast.literal_eval(pred.ASPECT_TERM)  # Convert string to list
        except Exception as e:
            print(f"Error parsing ASPECT_TERM: {e}")
            pred.ASPECT_TERM = []
            
    claude_pred.append(pred.ASPECT_TERM)
    print(pred.ASPECT_TERM)
    print('='*50 )


0
Yum!
Error parsing ASPECT_TERM: malformed node or string on line 1: <ast.Name object at 0x7f49496a1250>
[]
1
Serves really good sushi.
['sushi']
2
Not the biggest portions but adequate.
['portions']
3
Green Tea creme brulee is a must!
Error parsing ASPECT_TERM: invalid syntax (<unknown>, line 1)
[]
4
Don't leave the restaurant without it.
Error parsing ASPECT_TERM: malformed node or string on line 1: <ast.Name object at 0x7f494923e7d0>
[]
5
No Comparison
Error parsing ASPECT_TERM: malformed node or string on line 1: <ast.Name object at 0x7f494923e7d0>
[]
6
– I can't say enough about this place.
Error parsing ASPECT_TERM: malformed node or string on line 1: <ast.Name object at 0x7f4948162010>
[]
7
It has great sushi and even better service.
['sushi', 'service']
8
The entire staff was extremely accomodating and tended to my every need.
Error parsing ASPECT_TERM: malformed node or string on line 1: <ast.Name object at 0x7f494908a390>
[]
9
I've been to this restaurant over a dozen times 

In [29]:
ate_16_test['claude_pred'] = claude_pred

In [30]:
ate_16_test

,raw_text,aspectTerms,claude_pred
0,Yum!,"[{'term': 'noaspectterm', 'polarity': 'none'}]",[]
1,Serves really good sushi.,"[{'term': 'sushi', 'polarity': 'positive'}]",[sushi]
2,Not the biggest portions but adequate.,"[{'term': 'portions', 'polarity': 'neutral'}]",[portions]
3,Green Tea creme brulee is a must!,"[{'term': 'Green Tea creme brulee', 'polarity'...",[]
4,Don't leave the restaurant without it.,"[{'term': 'noaspectterm', 'polarity': 'none'}]",[]
...,...,...,...
671,Two rascally kids were seated near us for the ...,"[{'term': 'noaspectterm', 'polarity': 'none'}]",[]
672,Given that Ray's is a seafood restaurant...wel...,"[{'term': 'noaspectterm', 'polarity': 'none'}]",[]
673,"All considered, I have to say that Ray's Boath...","[{'term': ""Ray's Boathouse"", 'polarity': 'posi...",[Ray's Boathouse]
674,While I could have done without the youth who ...,"[{'term': 'server', 'polarity': 'positive'}, {...","[server, food]"


In [ ]:
ate_16_test['aspectTerms'] = ate_16_test['aspectTerms'].apply(eval)

In [32]:
def extract_terms(aspect_terms_list):
    terms = []
    for aspect in aspect_terms_list:
        terms.append(aspect['term'])
    return terms

ate_16_test['label'] = ate_16_test['aspectTerms'].apply(extract_terms)

In [34]:
ate_16_test['claude_pred'] = ate_16_test['claude_pred'].apply(lambda x: ['noaspectterm'] if x == [] else x)
ate_16_test

,raw_text,aspectTerms,claude_pred,label
0,Yum!,"[{'term': 'noaspectterm', 'polarity': 'none'}]",[noaspectterm],[noaspectterm]
1,Serves really good sushi.,"[{'term': 'sushi', 'polarity': 'positive'}]",[sushi],[sushi]
2,Not the biggest portions but adequate.,"[{'term': 'portions', 'polarity': 'neutral'}]",[portions],[portions]
3,Green Tea creme brulee is a must!,"[{'term': 'Green Tea creme brulee', 'polarity'...",[noaspectterm],[Green Tea creme brulee]
4,Don't leave the restaurant without it.,"[{'term': 'noaspectterm', 'polarity': 'none'}]",[noaspectterm],[noaspectterm]
...,...,...,...,...
671,Two rascally kids were seated near us for the ...,"[{'term': 'noaspectterm', 'polarity': 'none'}]",[noaspectterm],[noaspectterm]
672,Given that Ray's is a seafood restaurant...wel...,"[{'term': 'noaspectterm', 'polarity': 'none'}]",[noaspectterm],[noaspectterm]
673,"All considered, I have to say that Ray's Boath...","[{'term': 'Ray's Boathouse', 'polarity': 'posi...",[Ray's Boathouse],[Ray's Boathouse]
674,While I could have done without the youth who ...,"[{'term': 'server', 'polarity': 'positive'}, {...","[server, food]","[server, food]"


In [44]:
def get_metrics( y_true, y_pred):
        total_pred = 0
        total_gt = 0
        tp = 0
        for gt, pred in zip(y_true, y_pred):
            gt_list = gt.split(', ')
            pred_list = pred.split(', ')
            total_pred+=len(pred_list)
            total_gt+=len(gt_list)
            for gt_val in gt_list:
                for pred_val in pred_list:
                    if pred_val in gt_val or gt_val in pred_val:
                        tp+=1
                        break
        p = tp/total_pred
        r = tp/total_gt
        return p, r, 2*p*r/(p+r)        

In [45]:
precision, recall, f1 = get_metrics(ate_16_test['label'].apply(lambda x: ', '.join(x)), ate_16_test['claude_pred'].apply(lambda x: ', '.join(x)))

In [46]:
print(f"Precision: {precision}")   
print(f"Recall: {recall}")
print(f"F1: {f1}")

Precision: 0.6953210010881393
Recall: 0.7319587628865979
F1: 0.7131696428571429


In [43]:
ate_16_test.to_csv("/home/s6moakba/Thesis/agent_practice/claude_performance_16_res.csv")